<a href="https://colab.research.google.com/github/Roopanshi-Marwaha/Fraud_Ring_Detection/blob/main/Fraud_Ring_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import networkx as nx
import os

# ============================================================
# STEP 1: File paths define
# ============================================================
# files ko left sidebar "Files" panel se drag-drop karke upload kiya hai, isliye woh seedha /content/ folder ke andar hain
# (so koi unzip ya Drive mount karne ki zaroorat nahi)
base_path = '/content/'

accounts_file=os.path.join(base_path, 'accounts.csv')
alerts_file=os.path.join(base_path, 'alerts.csv')
transactions_file=os.path.join(base_path, 'transactions.csv')

try:
    # ============================================================
    # STEP 2: CSV files ko pandas DataFrame mein load kara hai
    # ============================================================
    # read_csv() teeno files ko memory mein table (DataFrame) ki tarah load karega.
    # transactions.csv badi (~58MB) hai isliye
    # yeh step thoda time lega.

    df_accounts=pd.read_csv(accounts_file)
    df_alerts=pd.read_csv(alerts_file)
    df_transactions=pd.read_csv(transactions_file)

    print("\n--- DataFrames Loaded ---")
    print("df_accounts head:")
    print(df_accounts.head())
    # pehli 5 rows dikhata hai, sanity check ke liye
    print("\ndf_alerts head:")
    print(df_alerts.head())
    print("\ndf_transactions head:")
    print(df_transactions.head())

    # ============================================================
    # STEP 3: Graph banaya hai (NetworkX)
    # ============================================================
    # DiGraph = Directed Graph, kyunki transaction ek direction mein hoti hai (sender → receiver), dono taraf nahi.
    G=nx.DiGraph()

    # --- Nodes added (har account ek node hai) ---
    # Har account_id ke liye uska IS_FRAUD flag bhi node ke
    # attribute ke roop mein store kar rahe hain, taaki baad mein fraud accounts ko easily filter/highlight kar sake.
    for account_id in df_accounts['ACCOUNT_ID']:
        is_fraud=df_accounts.loc[
            df_accounts['ACCOUNT_ID'] == account_id, 'IS_FRAUD'
        ].iloc[0]
        G.add_node(account_id, is_fraud=is_fraud)

    # --- Edges added (har transaction ek edge hai) ---
    # Edge sender se receiver ki taraf jaata hai.
    # amount, timestamp, alert_id — yeh sab edge ke "attributes" ki tarah store ho rahe hain, taaki baad mein query kar sako
    # (e.g. "$10,000 se zyada wale transactions dikhao").
    for _, row in df_transactions.iterrows():
        sender = row['SENDER_ACCOUNT_ID']
        receiver = row['RECEIVER_ACCOUNT_ID']
        amount = row['TX_AMOUNT']
        timestamp = row['TIMESTAMP']
        alert_id = row['ALERT_ID']

        G.add_edge(
            sender, receiver,
            amount=amount,
            weight=amount, # networkx algorithms 'weight' attribute expect karte hain
            timestamp=timestamp,
            alert_id=alert_id
        )

    # ============================================================
    # STEP 4: Graph ka summary print (for sanity check)
    # ============================================================
    print("\n--- Graph Constructed ---")
    print(f"Number of nodes: {G.number_of_nodes()}") # total accounts
    print(f"Number of edges: {G.number_of_edges()}") # total transactions
    print("Example nodes (first 5):", list(G.nodes(data=True))[:5])
    print("Example edges (first 5):", list(G.edges(data=True))[:5])

    # ============================================================
    # STEP 5: Alerts data checking
    # ============================================================
    # Yeh sirf confirm karne ke liye hai ki alerts.csv sahi load  hui hai.
    # isse baad mein fraud "typology" (kaunsa pattern tha — layering, smurfing, etc.) cross-reference karenge.

    print("\nAlerts DataFrame information:")
    print(df_alerts.info())

except FileNotFoundError as e:
    print(f"Error: File nahi mili. Files panel mein check karo ki "
          f"accounts.csv, alerts.csv, transactions.csv sahi se upload hui hain. {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


--- DataFrames Loaded ---
df_accounts head:
   ACCOUNT_ID CUSTOMER_ID  INIT_BALANCE COUNTRY ACCOUNT_TYPE  IS_FRAUD  \
0           0         C_0        184.44      US            I     False   
1           1         C_1        175.80      US            I     False   
2           2         C_2        142.06      US            I     False   
3           3         C_3        125.89      US            I     False   
4           4         C_4        151.13      US            I     False   

   TX_BEHAVIOR_ID  
0               1  
1               1  
2               1  
3               1  
4               1  

df_alerts head:
   ALERT_ID ALERT_TYPE  IS_FRAUD  TX_ID  SENDER_ACCOUNT_ID  \
0       193     fan_in      True     82               6976   
1       377      cycle      True    949               5776   
2       189     fan_in      True   6280               9999   
3       377      cycle      True   7999               1089   
4       130     fan_in      True  12975               7025   

